# Exploratory correlation analysis

Exploration only — the source of truth for metrics is
`experiments/runs/*/metrics.json` and `results/SUMMARY.md`.

Run this notebook from `GroupA/daniela`.

This notebook reuses `src.config` and `src.features` instead of duplicating the
data-loading and column-detection logic.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import data_dir, get_id_columns, get_targets, load_config
from src.features import detect_columns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

cfg = load_config()
TARGETS = get_targets(cfg)
ID_COLUMNS = get_id_columns(cfg)
print("Project root:", cfg["_root"])
print("Targets:", TARGETS)


In [ ]:
def load_table(source: str):
    path = data_dir(cfg) / source / "merged_dataset_features.csv"
    df = pd.read_csv(path, sep="\t")
    id_cols, wppsi_cols, feature_cols = detect_columns(df, ID_COLUMNS, TARGETS)
    for column in feature_cols + [t for t in TARGETS if t in df.columns]:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    return df, feature_cols

tables = {
    source: load_table(source)
    for source in ("clean_data_auto", "clean_data_manual")
}
for source, (df, features) in tables.items():
    print(f"{source}: {df.shape[0]} rows x {df.shape[1]} columns, {len(features)} features")


In [ ]:
def target_correlations(df, features, method="pearson"):
    return pd.DataFrame({
        target: df[features].corrwith(df[target], method=method)
        for target in TARGETS if target in df.columns
    })

auto_df, auto_features = tables["clean_data_auto"]
manual_df, manual_features = tables["clean_data_manual"]

corr = target_correlations(auto_df, auto_features)
corr.round(3).head(20)


In [ ]:
for target in corr.columns:
    top = corr[target].abs().sort_values(ascending=False).head(15)
    print(f"\nTop features for {target}")
    display(top.to_frame("abs_correlation"))


In [ ]:
top_features = corr.abs().max(axis=1).sort_values(ascending=False).head(25).index

fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(corr.loc[top_features], annot=True, fmt=".2f", cmap="vlag",
            center=0, ax=ax)
ax.set_title("Feature-target correlations (auto shift, top 25 features)")
fig.tight_layout()


In [ ]:
def linear_operation_correlations(df, features, target, max_features=6):
    """Correlate simple sums/differences/ratios of top features with one target."""
    top = corr[target].abs().sort_values(ascending=False).head(max_features).index.tolist()
    operations = {}
    for i, first in enumerate(top):
        for second in top[i + 1:]:
            operations[f"{first} + {second}"] = df[first] + df[second]
            operations[f"{first} - {second}"] = df[first] - df[second]
            operations[f"{first} / {second}"] = df[first] / df[second].replace(0, np.nan)
    operations = pd.DataFrame(operations)
    return operations.corrwith(df[target]).sort_values(key=np.abs, ascending=False)

for target in TARGETS:
    if target not in auto_df.columns:
        continue
    print(f"\nTop linear operations for {target}")
    display(linear_operation_correlations(auto_df, auto_features, target).head(15).to_frame("correlation"))


In [ ]:
delta = (auto_df[auto_features].corr().abs()
         - manual_df[manual_features].corr().abs()).abs()
print("Largest feature-feature correlation changes between auto and manual shift:")
delta.max(axis=1).sort_values(ascending=False).head(15).to_frame("max_abs_delta")


In [ ]:
out_dir = Path("results")
out_dir.mkdir(exist_ok=True)
corr.round(4).to_csv(out_dir / "correlation_features_vs_targets_auto.csv")
target_correlations(manual_df, manual_features).round(4).to_csv(
    out_dir / "correlation_features_vs_targets_manual.csv")
print("Saved small correlation summaries in", out_dir)
